# BCG X — Feature Engineering
## PowerCo Churn Investigation

This notebook uses the cleaned customer-level dataset provided for Task 3 and follows the feature-engineering framework in the supplied notebook.

**Note:** The cleaned dataset available for this task does not contain the historical `price_date` and historical monthly price columns required to reproduce Estelle's December–January feature. Therefore, that feature is not fabricated here. The available date and customer variables are used for the remaining feature-engineering steps.

## 1. Import packages

In [ ]:
import pandas as pd
import numpy as np

## 2. Load cleaned data

In [ ]:
data = pd.read_csv('./clean_data_after_eda.csv')
data.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000003,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,0.000011,2.896760e-06,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0


## 3. Review the dataset

In [ ]:
print('Shape:', data.shape)
print('\nData types:')
display(data.dtypes)
print('\nUnique values:')
display(data.nunique().sort_values())

Shape: (14606, 44)

Data types:


,0
id,object
channel_sales,object
cons_12m,int64
cons_gas_12m,int64
cons_last_month,int64
date_activ,object
date_end,object
date_modif_prod,object
date_renewal,object
forecast_cons_12m,float64



Unique values:


,0
has_gas,2
churn,2
origin_up,6
channel_sales,8
nb_prod_act,10
forecast_discount_energy,12
num_years_antig,13
forecast_price_pow_off_peak,41
var_6m_price_mid_peak_fix,105
var_6m_price_peak_fix,110


## 4. Remove columns with no useful variation

Columns containing only one unique value do not provide information for a churn model and can be removed.

In [ ]:
single_value_columns = data.columns[data.nunique() <= 1].tolist()
print('Columns with one or fewer unique values:')
print(single_value_columns)
data = data.drop(columns=single_value_columns)
print('\nNew shape:', data.shape)

Columns with one or fewer unique values:
[]

New shape: (14606, 44)


## 5. Estelle's suggested feature — data limitation

The starter notebook expects a historical pricing dataset containing `price_date` and historical off-peak price columns. The current `clean_data_after_eda.csv` is a customer-level dataset and contains no `price_date` column or historical monthly price observations. Therefore, the December/January feature cannot be calculated from this file without inventing data.

In [ ]:
required_historical_price_columns = ['price_date', 'off_peak_energy_price', 'off_peak_power_price']
missing_required_columns = [c for c in required_historical_price_columns if c not in data.columns]
print('Required historical pricing columns not available in cleaned dataset:')
print(missing_required_columns)

Required historical pricing columns not available in cleaned dataset:
['price_date', 'off_peak_energy_price', 'off_peak_power_price']


## 6. Create date-based features

Date information can be expanded into individual components that can be used as model features.

In [ ]:
date_columns = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']

for column in date_columns:
    data[column] = pd.to_datetime(data[column], errors='coerce')

for column in date_columns:
    data[f'{column}_year'] = data[column].dt.year
    data[f'{column}_month'] = data[column].dt.month
    data[f'{column}_day'] = data[column].dt.day

date_feature_columns = [
    c for c in data.columns
    if any(c.startswith(d + '_') for d in date_columns)
]

display(data[date_feature_columns].head())
print('Date features created:')
print(date_feature_columns)

,date_activ_year,date_activ_month,date_activ_day,date_end_year,date_end_month,date_end_day,date_modif_prod_year,date_modif_prod_month,date_modif_prod_day,date_renewal_year,date_renewal_month,date_renewal_day
0,2013,6,15,2016,6,15,2015,11,1,2015,6,23
1,2009,8,21,2016,8,30,2009,8,21,2015,8,31
2,2010,4,16,2016,4,16,2010,4,16,2015,4,17
3,2010,3,30,2016,3,30,2010,3,30,2015,3,31
4,2010,1,13,2016,3,7,2010,1,13,2015,3,9


Date features created:
['date_activ_year', 'date_activ_month', 'date_activ_day', 'date_end_year', 'date_end_month', 'date_end_day', 'date_modif_prod_year', 'date_modif_prod_month', 'date_modif_prod_day', 'date_renewal_year', 'date_renewal_month', 'date_renewal_day']


## 7. Combine related client variables

Consumption variables can be combined into simple features representing overall consumption and the relationship between electricity and gas consumption.

In [ ]:
data['total_cons_12m'] = data['cons_12m'] + data['cons_gas_12m']

data['gas_share_12m'] = np.where(
    data['total_cons_12m'] != 0,
    data['cons_gas_12m'] / data['total_cons_12m'],
    0
)

display(data[['cons_12m', 'cons_gas_12m', 'total_cons_12m', 'gas_share_12m']].head())

,cons_12m,cons_gas_12m,total_cons_12m,gas_share_12m
0,0,54946,54946,1.0
1,4660,0,4660,0.0
2,544,0,544,0.0
3,1584,0,1584,0.0
4,4425,0,4425,0.0


## 8. Review engineered features

In [ ]:
engineered_columns = date_feature_columns + ['total_cons_12m', 'gas_share_12m']
display(data[engineered_columns].head())
print('Number of engineered features:', len(engineered_columns))

,date_activ_year,date_activ_month,date_activ_day,date_end_year,date_end_month,date_end_day,date_modif_prod_year,date_modif_prod_month,date_modif_prod_day,date_renewal_year,date_renewal_month,date_renewal_day,total_cons_12m,gas_share_12m
0,2013,6,15,2016,6,15,2015,11,1,2015,6,23,54946,1.0
1,2009,8,21,2016,8,30,2009,8,21,2015,8,31,4660,0.0
2,2010,4,16,2016,4,16,2010,4,16,2015,4,17,544,0.0
3,2010,3,30,2016,3,30,2010,3,30,2015,3,31,1584,0.0
4,2010,1,13,2016,3,7,2010,1,13,2015,3,9,4425,0.0


Number of engineered features: 14


## 9. Final dataset review

In [ ]:
print('Final shape:', data.shape)
print('\nMissing values:')
display(data.isnull().sum().sort_values(ascending=False).head(20))
print('\nData types:')
display(data.dtypes)

Final shape: (14606, 58)

Missing values:


,0
id,0
channel_sales,0
cons_12m,0
cons_gas_12m,0
cons_last_month,0
date_activ,0
date_end,0
date_modif_prod,0
date_renewal,0
forecast_cons_12m,0



Data types:


,0
id,object
channel_sales,object
cons_12m,int64
cons_gas_12m,int64
cons_last_month,int64
date_activ,datetime64[ns]
date_end,datetime64[ns]
date_modif_prod,datetime64[ns]
date_renewal,datetime64[ns]
forecast_cons_12m,float64


## 10. Save the feature-engineered dataset

In [ ]:
data.to_csv('./feature_engineered_data.csv', index=False)
print('Saved as feature_engineered_data.csv')

Saved as feature_engineered_data.csv
